Pulled this from https://docu.ngsolve.org/nightly/i-tutorials/unit-2.1.1-preconditioners/preconditioner.html#Multigrid-preconditioner

In [16]:
# Import any packages needed
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np

In [17]:
def SolveProblem(h=0.5, p=1, levels=1,
                 condense=False,
                 precond="local"):
    """
    Solve Poisson problem on l refinement levels.
    PARAMETERS:
        h: coarse mesh size
        p: polynomial degree
        l: number of refinement levels
        condense: if true, perform static condensations
        precond: name of a built-in preconditioner
    Returns: (ndof, niterations)
        List of tuples of number of degrees of freedom and iterations
    """
    mesh = Mesh(unit_square.GenerateMesh(maxh=h))
    fes = H1(mesh, order=p, dirichlet="left|right")

    u, v = fes.TnT()
    k = CoefficientFunction(1+x*x+y*y)
    a = BilinearForm(k*grad(u)*grad(v)*dx+0.25*u*v*dx, condense=condense)
    f = LinearForm(v*dx)
    gfu = GridFunction(fes)
    w = 10
    s = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/w)*sin(w*pi*x)*sin(w*pi*y))
    gfu.Set(s)
    Draw(gfu)
    c = Preconditioner(a, precond) # Register c to a BEFORE assembly

    steps = []

    for l in range(levels):
        if l > 0:
            mesh.Refine()
        fes.Update()
        gfu.Update()

        with TaskManager():
            a.Assemble()
            f.Assemble()

            # Conjugate gradient solver
            inv = CGSolver(a.mat, c.mat, maxsteps=1000)

            # Solve steps depend on condense
            if condense:
                f.vec.data += a.harmonic_extension_trans * f.vec

            gfu.vec.data = inv * f.vec
            Draw(gfu)
            
            if condense:
                gfu.vec.data += a.harmonic_extension * gfu.vec
                gfu.vec.data += a.inner_solve * f.vec
        steps.append ( (fes.ndof, inv.GetSteps()) )
        if fes.ndof < 15000:
            Redraw()
    return steps, gfu

In [20]:
res_mg,gssol = SolveProblem(levels=2, precond="multigrid")
res_mg

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

[(8, 2), (21, 4)]

In [21]:
Draw(gssol)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene